In [0]:
%sql
USE CATALOG workspace;
USE SCHEMA default;

In [0]:
%sql
SELECT 
  'NULL Check' as check_type,
  COUNT(*) as total_rows,
  SUM(CASE WHEN game_name IS NULL THEN 1 ELSE 0 END) as null_game_name,
  SUM(CASE WHEN genre IS NULL THEN 1 ELSE 0 END) as null_genre,
  SUM(CASE WHEN rank_type IS NULL THEN 1 ELSE 0 END) as null_rank_type,
  SUM(CASE WHEN rank IS NULL THEN 1 ELSE 0 END) as null_rank
FROM bronze_rankings;

In [0]:
%sql
SELECT 'Genres' as category, genre as value, COUNT(*) as count
FROM bronze_rankings
GROUP BY genre

In [0]:
%sql
SELECT COUNT(*)
FROM bronze_rankings

In [0]:
%sql
SELECT COUNT(DISTINCT(game_name))
FROM bronze_rankings

In [0]:
display(spark.sql("SELECT * FROM bronze_rankings LIMIT 5"))

In [0]:
%sql
-- Data quality checks for bronze_rankings

-- 1. Check for NULL values
SELECT 
  'NULL Check' as check_type,
  COUNT(*) as total_rows,
  SUM(CASE WHEN game_name IS NULL THEN 1 ELSE 0 END) as null_game_name,
  SUM(CASE WHEN genre IS NULL THEN 1 ELSE 0 END) as null_genre,
  SUM(CASE WHEN rank_type IS NULL THEN 1 ELSE 0 END) as null_rank_type,
  SUM(CASE WHEN rank IS NULL THEN 1 ELSE 0 END) as null_rank
FROM bronze_rankings;

-- 2. Check for empty strings
SELECT 
  'Empty String Check' as check_type,
  SUM(CASE WHEN TRIM(game_name) = '' THEN 1 ELSE 0 END) as empty_game_name,
  SUM(CASE WHEN TRIM(genre) = '' THEN 1 ELSE 0 END) as empty_genre,
  SUM(CASE WHEN TRIM(rank_type) = '' THEN 1 ELSE 0 END) as empty_rank_type
FROM bronze_rankings;

-- 3. Check distinct values for categorical columns
SELECT 'Genres' as category, genre as value, COUNT(*) as count
FROM bronze_rankings
GROUP BY genre
UNION ALL
SELECT 'Rank Types' as category, rank_type as value, COUNT(*) as count
FROM bronze_rankings
GROUP BY rank_type
ORDER BY category, count DESC;

-- 4. Check for duplicate rankings within same genre and rank_type
SELECT 
  genre,
  rank_type,
  rank,
  COUNT(*) as duplicate_count,
  COLLECT_LIST(game_name) as games
FROM bronze_rankings
GROUP BY genre, rank_type, rank
HAVING COUNT(*) > 1
ORDER BY genre, rank_type, rank;

-- 5. Check rank continuity (gaps in ranking)
WITH ranked_data AS (
  SELECT 
    genre,
    rank_type,
    rank,
    ROW_NUMBER() OVER (PARTITION BY genre, rank_type ORDER BY rank) as expected_rank
  FROM bronze_rankings
)
SELECT 
  genre,
  rank_type,
  COUNT(*) as total_ranks,
  MAX(rank) as max_rank,
  SUM(CASE WHEN rank != expected_rank THEN 1 ELSE 0 END) as gaps_or_duplicates
FROM ranked_data
GROUP BY genre, rank_type
ORDER BY genre, rank_type;

-- 6. Check for games appearing in multiple genres
SELECT 
  game_name,
  rank_type,
  COUNT(DISTINCT genre) as genre_count,
  COLLECT_LIST(DISTINCT genre) as genres
FROM bronze_rankings
GROUP BY game_name, rank_type
HAVING COUNT(DISTINCT genre) > 1
ORDER BY genre_count DESC, game_name;

In [0]:
%sql
-- Games in rankings but NOT in silver_games
SELECT COUNT(DISTINCT game_name) 
FROM bronze_rankings
WHERE game_name NOT IN (SELECT DISTINCT name FROM silver_games);

-- Should show 17 (303 - 286)

In [0]:
%sql
SELECT DISTINCT game_name
FROM bronze_rankings
WHERE game_name NOT IN (SELECT DISTINCT name FROM silver_games)
LIMIT 10;